# Hab Dishab DA During Investigation, Full Bout, and Non-Investigation

This notebook computes mean DA within each Hab/Dishab social bout for three windows:

- `full_bout`: introduction to removal
- `investigation`: only frames inside `Investigation` annotations
- `non_investigation`: the rest of the bout outside `Investigation`

It is set up to run both `mPFC` and `NAc`, and the plots use boxplots plus connected individual-mouse trajectories across bouts.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'Hab_Dishab' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from experiment_class import Experiment

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [ ]:
REGION_CONFIGS = {
    'mPFC': {
        'experiment_path': r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Hab_Dishab\All\mpfc",
        'csv_path': r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Hab_Dishab\All\mpfc_csvs",
        'color': '#FFAF00',
    },
    'NAc': {
        'experiment_path': r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Hab_Dishab\All\nac",
        'csv_path': r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Hab_Dishab\All\nac_csvs",
        'color': '#15616F',
    },
}

SELECTED_REGIONS = ['mPFC', 'NAc']

BOUT_DEFINITIONS = [
    {'prefix': 's1', 'introduced': 's1_Introduced', 'removed': 's1_Removed'},
    {'prefix': 's2', 'introduced': 's2_Introduced', 'removed': 's2_Removed'},
]

BOUT_ORDER = ['s1-1', 's1-2', 's1-3', 's1-4', 's1-5', 's2-1']
WINDOW_ORDER = ['full_bout', 'investigation', 'non_investigation']
SIGNAL_NAME = 'zscore'

OUTPUT_DIR = PROJECT_ROOT / 'Hab_Dishab' / 'investigation_full_noninvestigation_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

In [ ]:
def load_region_experiment(region_name):
    config = REGION_CONFIGS[region_name]
    exp = Experiment(config['experiment_path'], config['csv_path'])
    exp.default_batch_process()
    exp.group_extract_manual_annotations(BOUT_DEFINITIONS, first_only=False)
    return exp


def get_signal_array(trial, signal_name='zscore'):
    if not hasattr(trial, signal_name):
        raise AttributeError(f"Trial does not have signal '{signal_name}'")
    return np.asarray(getattr(trial, signal_name), dtype=float)


def merge_intervals(intervals):
    cleaned = sorted(
        (float(start), float(end))
        for start, end in intervals
        if pd.notna(start) and pd.notna(end) and end > start
    )
    if not cleaned:
        return []

    merged = [cleaned[0]]
    for start, end in cleaned[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged


def subtract_intervals(base_interval, intervals):
    base_start, base_end = map(float, base_interval)
    remaining = [(base_start, base_end)]

    for cut_start, cut_end in merge_intervals(intervals):
        updated = []
        for seg_start, seg_end in remaining:
            if cut_end <= seg_start or cut_start >= seg_end:
                updated.append((seg_start, seg_end))
                continue
            if cut_start > seg_start:
                updated.append((seg_start, cut_start))
            if cut_end < seg_end:
                updated.append((cut_end, seg_end))
        remaining = updated

    return [(start, end) for start, end in remaining if end > start]


def intervals_mean(timestamps, signal, intervals):
    pooled = []
    total_n = 0
    for start, end in intervals:
        mask = (timestamps >= start) & (timestamps <= end)
        values = signal[mask]
        values = values[~np.isnan(values)]
        if values.size == 0:
            continue
        pooled.append(values)
        total_n += int(values.size)

    if not pooled:
        return np.nan, 0

    pooled = np.concatenate(pooled)
    return float(np.nanmean(pooled)), total_n


def build_window_dataframe(experiment, region_name, signal_name='zscore'):
    rows = []
    for trial_name, trial in experiment.trials.items():
        timestamps = np.asarray(trial.timestamps, dtype=float)
        signal = get_signal_array(trial, signal_name)
        behaviors = trial.behaviors.copy()

        for bout_label, (bout_start, bout_end) in trial.bouts.items():
            bout_df = behaviors[(behaviors['Bout'] == bout_label) & (behaviors['Behavior'] == 'Investigation')].copy()
            investigation_intervals = merge_intervals(list(zip(bout_df['Event_Start'], bout_df['Event_End'])))
            full_bout_intervals = [(float(bout_start), float(bout_end))]
            non_investigation_intervals = subtract_intervals((bout_start, bout_end), investigation_intervals)

            for window_name, intervals in [
                ('full_bout', full_bout_intervals),
                ('investigation', investigation_intervals),
                ('non_investigation', non_investigation_intervals),
            ]:
                mean_signal, n_samples = intervals_mean(timestamps, signal, intervals)
                rows.append(
                    {
                        'BrainRegion': region_name,
                        'trial_name': trial_name,
                        'Subject': trial.subject_name,
                        'Bout': bout_label,
                        'Window': window_name,
                        'mean_signal': mean_signal,
                        'n_samples': n_samples,
                        'window_duration_s': float(sum(end - start for start, end in intervals)),
                        'signal_name': signal_name,
                    }
                )

    out = pd.DataFrame(rows)
    out['Bout'] = pd.Categorical(out['Bout'], categories=BOUT_ORDER, ordered=True)
    out['Window'] = pd.Categorical(out['Window'], categories=WINDOW_ORDER, ordered=True)
    return out.sort_values(['BrainRegion', 'Window', 'Bout', 'Subject']).reset_index(drop=True)


def build_all_regions_dataframe(region_names, signal_name='zscore'):
    parts = []
    for region_name in region_names:
        exp = load_region_experiment(region_name)
        parts.append(build_window_dataframe(exp, region_name=region_name, signal_name=signal_name))
    return pd.concat(parts, ignore_index=True)


def summarize_windows(window_df):
    summary_by_bout = (
        window_df.groupby(['BrainRegion', 'Bout', 'Window'], observed=True)['mean_signal']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    summary_by_bout['sem'] = summary_by_bout['std'] / np.sqrt(summary_by_bout['count'].clip(lower=1))

    summary_overall = (
        window_df.groupby(['BrainRegion', 'Window'], observed=True)['mean_signal']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    summary_overall['sem'] = summary_overall['std'] / np.sqrt(summary_overall['count'].clip(lower=1))
    return summary_by_bout, summary_overall


def plot_boxplots_with_subject_lines(window_df, region_names=None, figsize_per_panel=(4.4, 4.0)):
    plot_df = window_df.dropna(subset=['mean_signal', 'Bout']).copy()
    plot_df['Bout_str'] = plot_df['Bout'].astype(str)
    plot_df = plot_df[plot_df['Bout_str'].isin(BOUT_ORDER)].copy()
    bout_to_x = {bout: idx for idx, bout in enumerate(BOUT_ORDER)}

    if region_names is None:
        region_names = list(plot_df['BrainRegion'].dropna().unique())

    n_rows = len(region_names)
    n_cols = len(WINDOW_ORDER)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1] * n_rows),
        sharey='row',
        squeeze=False,
    )

    for row_idx, region_name in enumerate(region_names):
        region_color = REGION_CONFIGS[region_name]['color']
        for col_idx, window_name in enumerate(WINDOW_ORDER):
            ax = axes[row_idx, col_idx]
            sub = plot_df[(plot_df['BrainRegion'] == region_name) & (plot_df['Window'] == window_name)].copy()

            if sub.empty:
                ax.set_axis_off()
                continue

            sns.boxplot(
                data=sub,
                x='Bout_str',
                y='mean_signal',
                order=BOUT_ORDER,
                color=region_color,
                width=0.6,
                fliersize=0,
                linewidth=1.1,
                ax=ax,
            )

            for _, subject_df in sub.groupby('Subject', observed=True):
                subject_df = subject_df.sort_values('Bout')
                x_vals = [bout_to_x[bout] for bout in subject_df['Bout_str'] if bout in bout_to_x]
                y_vals = subject_df.loc[subject_df['Bout_str'].isin(bout_to_x), 'mean_signal'].to_numpy(dtype=float)
                if len(x_vals) == 0:
                    continue
                ax.plot(x_vals, y_vals, color='0.45', alpha=0.45, linewidth=1.0, zorder=2)
                ax.scatter(
                    x_vals,
                    y_vals,
                    s=28,
                    color=region_color,
                    edgecolor='black',
                    linewidth=0.4,
                    alpha=0.9,
                    zorder=3,
                )

            ax.set_title(f'{region_name} | {window_name}')
            ax.set_xlabel('Bout')
            if col_idx == 0:
                ax.set_ylabel(f'Mean {SIGNAL_NAME}')
            else:
                ax.set_ylabel('')
            ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()


def plot_region_window_means(summary_overall):
    plt.figure(figsize=(7, 5))
    sns.barplot(
        data=summary_overall,
        x='Window',
        y='mean',
        hue='BrainRegion',
        order=WINDOW_ORDER,
        palette={region: REGION_CONFIGS[region]['color'] for region in REGION_CONFIGS},
        errorbar=None,
    )
    plt.ylabel(f'Mean {SIGNAL_NAME}')
    plt.xlabel('Window')
    plt.title(f'Overall mean {SIGNAL_NAME} by region and window')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
window_df = build_all_regions_dataframe(SELECTED_REGIONS, signal_name=SIGNAL_NAME)
summary_by_bout, summary_overall = summarize_windows(window_df)

window_df.to_csv(OUTPUT_DIR / f'{SIGNAL_NAME}_window_means_by_subject_all_regions.csv', index=False)
summary_by_bout.to_csv(OUTPUT_DIR / f'{SIGNAL_NAME}_window_summary_by_bout_all_regions.csv', index=False)
summary_overall.to_csv(OUTPUT_DIR / f'{SIGNAL_NAME}_window_summary_overall_all_regions.csv', index=False)

window_df.head(12)

In [ ]:
summary_by_bout

In [ ]:
summary_overall

## Boxplots With Individual Mice Connected Across Bouts

In [ ]:
plot_boxplots_with_subject_lines(window_df, region_names=SELECTED_REGIONS)

## Optional Overall Region Summary

In [ ]:
plot_region_window_means(summary_overall)

In [ ]:
region_window_pivot = (
    window_df.pivot_table(
        index=['BrainRegion', 'Subject', 'Bout'],
        columns='Window',
        values='mean_signal',
        observed=True,
    )
    .reset_index()
)
region_window_pivot.head()

## Notes

- Consecutive `Investigation` events within 1 second are merged by the existing pipeline before averaging.
- In the boxplot panels, each line is one mouse tracked across bouts within the same window.
- If a mouse has no investigation time in a bout, the `investigation` value is left as `NaN`.